# Hydraulisch-thermische Netzstudie: Großwärmespeicher MAN/UPM im Fernwärmenetz Stadtbach

**Stand:** Etappe 1 von 4 (Kapitel 1–8: Daten, Qualität, Heizkurve). Netzmodell, Optimierung, Szenarien und pandapipes folgen in den nächsten Etappen.

## 1. Management Summary
> *Wird am Ende des Workflows auf Basis der berechneten Ergebnisse erzeugt (Kapitel 22–23). Bis dahin gilt: keine belastbare Aussage zur Speicherwirkung.*

**Bisherige Zwischenbefunde (Etappe 1):** siehe Abschnitte 5.2 (Widersprüche), 6.3 (Datenqualität) und 8.4 (Heizkurve).

## 2. Zielsetzung und Systemgrenzen
**Frage:** Kann ein direkt eingebundener, druckbeaufschlagter Heißwasserspeicher bei MAN/UPM kritische Druck- und Temperatursituationen im bestehenden und ausgebauten Netz vermeiden oder reduzieren – und welche Speicher-, Pumpen-, Anschluss- und Betriebsparameter sind dafür nötig?

**Systemgrenze:** Fernwärmenetz Stadtbach (Netz Mitte/Ost/Süd/West) mit Erzeugern HKW, GT-Ost, BMHKW, AVA, Heizwerk Süd/West; Speicher ohne trennenden Wärmeübertrager.

**Grundsatz Speicherphysik:** Ein direkt eingebundener Speicher erzeugt *keine* Förderhöhe. Er kann Druckverluste nur über lokale Einspeisung und reduzierte Transportströme senken. Aktive Druckanhebung erfordert eine Pumpe. Diese Unterscheidung wird in Modell und Ergebnisdarstellung getrennt ausgewiesen.

**Arbeitsannahmen (mit dem Auftraggeber abgestimmt):** (i) Speicher als *Druckspeicher* wegen bis zu 120 °C im Winter; (ii) MAN-Anschlusspunkt wird als Annahme aus den Plänen festgelegt und als Sensitivität variiert; (iii) lineare Vorlauf-Heizkurve 120 °C bei −10 °C bis 90 °C bei +20 °C als Vorgabe.

## 3. Reproduzierbarkeit und Softwareumgebung

In [ ]:
import os, sys, re, json, math, platform, hashlib, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ---------------- zentrale Konfiguration ----------------
SEED = 20250101
np.random.seed(SEED)

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "calion").is_dir() and (p / "configs").is_dir():
            return p
    raise FileNotFoundError("Repository-Wurzel (Ordner 'calion' + 'configs') nicht gefunden.")

ROOT      = find_repo_root()
DATA_DIR  = ROOT / "data" / "Stadtbach" / "upload"      # NDA: nicht versionieren!
RESULTS   = ROOT / "results" / "studie_MAN_UPM"
FIG_DIR   = RESULTS / "abbildungen"
for d in (RESULTS, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    year=2025,
    p_unit="bar (Überdruck, Annahme – siehe Annahmenkatalog A-01)",
    heizkurve_vorgabe=dict(T_low_out=-10.0, T_VL_at_low=120.0, T_high_out=20.0, T_VL_at_high=90.0),
    cp_kJ_kgK=4.19, rho_kg_m3=971.8,        # Näherung bei ~80 °C; später temperaturabhängig
    dt_h=1.0,
)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = dict(vl="#c0392b", rl="#2874a6", ok="#1e8449", warn="#d68910", grau="#7f8c8d", mess="#34495e")

print("Python", platform.python_version(), "|", platform.platform())
for m in ("numpy","pandas","scipy","matplotlib","pyarrow","openpyxl","yaml","pyomo","pandapipes"):
    try:
        mod = __import__(m); print(f"{m:11s}", getattr(mod, "__version__", "?"))
    except Exception as e:
        print(f"{m:11s} NICHT installiert ({type(e).__name__})")
print("Repo-Wurzel:", ROOT.name, "| Datenordner vorhanden:", DATA_DIR.exists())

## 4. Repository- und Dateiinventur
Rekursive Bestandsaufnahme der für die Studie relevanten Verzeichnisse. Die Bedeutung wird – wo möglich – am **Inhalt** geprüft (YAML: Knoten/Leitungen; Parquet: Spalten/Zeilen; PDF: Seitenzahl), nicht am Dateinamen.
Schreibweise „Stadtbach": tolerante, case-insensitive Suche (der Auftrag nennt `stadtabcach`).

In [ ]:
import yaml
# --- 4.1 Tolerante Ordnersuche
def fuzzy_dirs(base, pattern=r"stadt.?a?b?a?c?h"):
    rx = re.compile(pattern, re.I)
    return sorted({str(p.relative_to(ROOT)) for p in base.rglob("*")
                   if p.is_dir() and rx.search(p.name) and ".git" not in p.parts})
print("Gefundene Stadtbach-Ordner:")
for d in fuzzy_dirs(ROOT): print("  ", d)

# --- 4.2 Inventur
SCAN = [ROOT/"configs", ROOT/"docs", ROOT/"paper2_package", ROOT/"scripts", ROOT/"tools", ROOT/"calion",
        ROOT/"revision", DATA_DIR]
EXT = {".yaml",".yml",".csv",".parquet",".json",".geojson",".shp",".dxf",".dwg",".pdf",".xlsx",".xls",
       ".ipynb",".md",".docx",".pptx",".py"}
rows=[]
for base in SCAN:
    if not base.exists(): continue
    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXT and ".git" not in p.parts and "_build" not in p.parts:
            rel = p.relative_to(ROOT).as_posix()
            if not re.search(r"stadt|WV6|640|acron|Import_Data|storage_geometry|pandapipes|Lastg", rel, re.I):
                continue
            st = p.stat()
            rows.append(dict(pfad=rel, typ=p.suffix.lower().lstrip("."), groesse_kB=round(st.st_size/1024,1),
                             geaendert=datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d")))
inv = pd.DataFrame(rows).sort_values("pfad").reset_index(drop=True)
print(len(inv), "relevante Dateien")

In [ ]:
# --- 4.3 Inhaltsprüfung + Bewertung
def inspect(rel):
    p = ROOT/rel; ext = p.suffix.lower()
    try:
        if ext in (".yaml",".yml"):
            d = yaml.safe_load(p.read_text(encoding="utf-8", errors="ignore"))
            n = d.get("network", {}) if isinstance(d, dict) else {}
            if n.get("nodes") and n.get("pipes"):
                return f"YAML-Netz: {len(n['nodes'])} Knoten, {len(n['pipes'])} Leitungen"
            return "YAML (kein Netz)"
        if ext == ".parquet":
            df = pd.read_parquet(p); return f"Parquet {df.shape[0]}x{df.shape[1]}"
        if ext == ".csv":
            df = pd.read_csv(p, nrows=5, sep=None, engine="python"); return f"CSV, Spalten: {list(df.columns)[:4]}…"
        if ext == ".pdf":
            txt = p.read_bytes(); npg = len(re.findall(rb"/Type\s*/Page[^s]", txt)); return f"PDF, {npg} Seite(n), Raster (kein Text erwartet)"
        if ext == ".xlsx":
            xl = pd.ExcelFile(p); return f"Excel, Blätter: {xl.sheet_names[:3]}"
    except Exception as e:
        return f"Lesefehler: {type(e).__name__}"
    return ""

def bewerte(rel):
    r = rel.lower()
    if "upload/a_daten/acron_stundenwerte_wert" in r: return ("Druck/Temperatur/Durchfluss/Leistung 2025 (stündlich)", "verwendet", "Kalibrierung/Validierung, Heizkurve", "Zeitumstellung: 1 Doppelstunde verworfen; viele Lücken")
    if "upload/a_daten/acron_stundenwerte_minmax" in r: return ("Min/Max zu ACRON-Reihen", "verwendet (später)", "Druckspitzen/-minima", "nicht in Etappe 1 ausgewertet")
    if "acron_signalindex" in r: return ("Zuordnung Signal -> Originaldatei/Station", "verwendet", "Messstellenzuordnung", "-")
    if "import_data_stadtbach_15min" in r: return ("Außentemperatur, V_n-Zähler, WRG (15 min)", "verwendet", "Außentemperatur, Kundenlasten", "Einheiten der V_n-Kanäle ungeklärt")
    if "combined" in r: return ("Aggregierte Kundenlasten (7 gemessen + 17 geschätzt)", "nur Referenz", "Vergleich zur alten Config", "17 Reihen zonal geschätzt -> keine Validierungsziele")
    if "b_plaene" in r: return ("Netzpläne WV640/650/660 (Raster)", "verwendet (Etappe 2)", "Topologie, Armaturen, Höhen, PN", "verkleinerte Raster; Text nicht extrahierbar")
    if "lastg" in r: return ("Lastgänge 2020–2022", "nicht verwendet (Etappe 1)", "evtl. Ausbau-/Lastform", "älter als 2025")
    if r.endswith("stadtbach_topo.yaml") and "configs/paper_2/" in r and "archiev" not in r: return ("Aktuelles YAML-Netz (Baum)", "verwendet als Ausgangspunkt", "Sektions-Skelett, Erzeuger", "Baum statt Masche; Druckniveau nicht messkonform")
    if "archiev" in r and "stadtbach" in r: return ("Ältere Config-Stände", "nicht verwendet", "-", "veraltet; Kommentare widersprüchlich")
    if "storage_geometry" in r: return ("Speichergeometrie (atmosphärische Grube)", "nicht verwendet", "Widerspruch zum Druckspeicher", "atmosph. 95 °C-Grenze; nicht passend")
    if "20250922_stadtbach.ipynb" in r: return ("Alter Pyomo-MILP (keine pandapipes)", "nicht verwendet", "-", "keine pandapipes-Studie vorhanden")
    if "pandapipes" in r: return ("pandapipes-Crosscheck (Memmingen)", "Vorlage", "Vorgehen für Kontrolle", "nur Memmingen")
    if r.endswith(".py") and "stadtbach" in r: return ("Hilfsskript", "nicht verwendet", "-", "-")
    return ("Sonstiges/Dokumentation", "nicht verwendet", "-", "-")

inv["inhalt_geprueft"] = inv.pfad.map(inspect)
b = inv.pfad.map(bewerte)
inv["bedeutung"], inv["status"], inv["begruendung"], inv["qualitaet"] = [b.map(lambda t: t[i]) for i in range(4)]
inv.to_csv(RESULTS/"inventur_dateien.csv", index=False)
with pd.option_context("display.max_colwidth", 70, "display.width", 250):
    display(inv[inv.status.str.startswith(("verwendet","nur","Vorlage"))][["pfad","typ","groesse_kB","inhalt_geprueft","status"]])
print("Status-Übersicht:", inv.status.value_counts().to_dict())

**Befund 4.4 – fehlende Quellen:** Suche nach Stadtbach-DXF/DWG/Shape/GeoJSON: siehe nächste Zelle. Die Pläne liegen nur als verkleinerte Raster-PDFs vor.

In [ ]:
geo = inv[inv.typ.isin(["dxf","dwg","shp","geojson"])]
print("Geometriedateien (Stadtbach):", "keine" if geo.empty else geo.pfad.tolist())
print("DXF im Repo (nur Memmingen erwartet):", [p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.dxf") if ".git" not in p.parts])
print("Stadtbach-pandapipes-Studie im Repo:", "nein – nur Memmingen-Crosscheck" if not any("pandapipes" in x and "stadtbach" in x.lower() for x in inv.pfad) else "ja")

## 5. Quellen-, Konflikt- und Annahmenkatalog

### 5.1 Annahmenkatalog
Herkunftsklassen: **M** direkt gemessen · **P** Plan/Betriebsunterlage · **Y** YAML/bestehendes Modell · **H** Hersteller · **K** kalibriert · **A** fachlicher Standardwert · **S** Sensitivitätsparameter. Der Katalog wird in späteren Etappen fortgeschrieben.

In [ ]:
katalog = pd.DataFrame([
 # id, parameter, wert, einheit, klasse, quelle, gueltig, unsicherheit, wirkung, sens_bereich
 ("A-01","Druckdefinition ACRON", "Überdruck", "bar", "A","Plausibilität: HKW-RL-Median 8,5 bar; Absolutdruck unwahrscheinlich niedrig für Ruhedruck 3,2–6,5 bar Ü", "2025","hoch: Faktor 1,013 bar","Siedesicherheit, Speicherdruck","abs vs. Ü (Δ=1,013 bar)"),
 ("A-02","Durchflusseinheit ACRON", "m³/h (≈ t/h)", "m³/h", "A","Leistungsbilanz MAN (Abschnitt 6.4)", "2025","mittel","Massenströme","t/h vs. m³/h (Δρ≈3–4 %)"),
 ("A-03","Heizkurve Hauptrechnung", "T_VL=110−T_a, begrenzt 90…120 °C", "°C", "S","Auftraggeber-Vorgabe (vorläufig)", "Winter/Übergang","mittel","Vorlauftemperatur","YAML-Kurve (122→70 °C) als Alternative"),
 ("A-04","Speichertyp", "Druckspeicher, direkt eingebunden", "-", "P","Auftraggeber: bis 120 °C im Winter nötig", "-","-","Betriebsdruck ≥ p_sat(120 °C)+Marge","-"),
 ("A-05","MAN-Anschlusspunkt", "wird in Etappe 2 aus WV650 festgelegt", "-", "A","Auftraggeber: 'Du kannst den MAN-Punkt annehmen'", "-","hoch","Speicherhydraulik","Varianten am Netzknoten"),
 ("A-06","Netz-Spreizung Dimensionierung", "99/60 °C", "°C", "Y","Stadtbach_topo.yaml", "Übergangszeit","mittel","Erzeugerkapazität","Messung 2025 (Abschnitt 8)"),
 ("A-07","Transportgrenzen SL5/SL6/SL7 (separiert)", "750/500/300 t/h", "t/h", "P","WV650 laut Auftrag (noch nicht am Plan geprüft)", "Stand 2022","mittel","Separierter Betrieb","±20 %"),
 ("A-08","Mindest-Ruhedruck bei 120 °C", "MITTE–CITY 6,5 / MITTE 3,2 / SÜD 3,2 / OST 2,5", "bar Ü", "P","WV650 laut Auftrag (noch nicht am Plan geprüft)", "Stand 2022","gering","Druckhaltung","-"),
 ("A-09","Kundendifferenzdruck minimal", "0,7", "bar", "Y","Stadtbach_topo.yaml delta_p_min_consumer_bar", "-","hoch (Messung 0–4 bar)","Druckverletzungen","0,3 … 1,5 bar"),
 ("A-10","Rohrrauheit", "0,5 (YAML) – wirkt dort nicht", "mm", "Y","Stadtbach_topo.yaml; im Framework nur gespeichert, nicht verwendet", "-","hoch","Druckverlust","0,1 … 1,0 mm (Colebrook im Notebook)"),
], columns=["id","parameter","wert","einheit","klasse","quelle_fundstelle","gueltigkeit","unsicherheit","auswirkung","sensitivitaetsbereich"])
katalog.to_csv(RESULTS/"annahmenkatalog.csv", index=False)
with pd.option_context("display.max_colwidth", 60, "display.width", 250): display(katalog[["id","parameter","wert","einheit","klasse","unsicherheit"]])

### 5.2 Konfliktkatalog (Widersprüche zwischen Quellen)
Keine Quelle wird stillschweigend überschrieben; für jeden Widerspruch wird eine Arbeitsannahme festgelegt und die Alternative als Sensitivität gerechnet.

In [ ]:
konflikte = pd.DataFrame([
 ("K-01","Heizkurve","Vorgabe: 120 °C@−10 °C … 90 °C@+20 °C","YAML: 122 °C@−15 °C … 70 °C@+30 °C (+Saisonwerte)","Abschnitt 8: Vergleich mit Messung","Vorgabe für Hauptrechnung, YAML als Sensitivität"),
 ("K-02","Netztopologie","Auftrag/Plan: teilweise vermascht, Sektionen","YAML: 33 Knoten/32 Leitungen, reiner Baum; Header nennt 32/31; Paper-Tabelle nennt 'meshed'","Etappe 2","Sektionsmodell mit Maschen aus WV650/660"),
 ("K-03","Speichertyp","Auftraggeber: Druckspeicher bis 120 °C","storage_geometry.yaml: atmosphärische Grube (95 °C, bis 120 000 m³); Topo-YAML: 10 bar/5 000 m³","Kap. 15","Druckspeicher; atmosphärische Variante entfällt"),
 ("K-04","Speicherenergie","5 000 m³ @ 10 K ≈ 56–58 MWh","YAML-Leiter bis 84,5 MWh (≈ 15 K)","Kap. 15","aus Volumen und ΔT berechnen; 84,5 MWh nicht übernehmen"),
 ("K-05","Druckniveau","Messung 2025: HKW-VL Median 10,7 bar, RL 8,5 bar","YAML: VL-Sollwert 16 bar, RL am Boden 0,5 bar","Kap. 13/14","Messwerte als Anker; YAML-Sollwert verwerfen"),
 ("K-06","Erzeugerleistungen (Kommentare)","YAML-Assets: HKW 127, HWS 32, HWW 104 MW","YAML-Header/Serienmap: HKW 75, HWS 40, HWW 64 MW","Etappe 2/3","Asset-Werte (Üzeit, 99/60 °C); Kommentare veraltet"),
 ("K-07","Hydraulik im Framework","Darcy-Weisbach mit Rauheit (Auftrag)","Framework: konstantes λ=0,02; Δp nur als untere Hüllkurve","Etappe 2","eigener Netzlöser im Notebook"),
], columns=["id","thema","quelle_A","quelle_B","pruefung","arbeitsannahme"])
konflikte.to_csv(RESULTS/"konfliktkatalog.csv", index=False)
with pd.option_context("display.max_colwidth", 55, "display.width", 250): display(konflikte)

## 6. Import und Prüfung der Messdaten
Rohdaten werden **nie überschrieben**: `acron_raw` bleibt unverändert, `acron` ist die bereinigte Arbeitskopie.
Geprüft werden Zeitstempel/Zeitzone/Sommerzeit, Duplikate, Lücken, Nullwerte, eingefrorene Werte, Ausreißer, Vorzeichen und Einheiten.

In [ ]:
# --- 6.1 Import
A = DATA_DIR/"A_Daten"
sig      = pd.read_csv(A/"acron_signalindex.csv")
acron_raw= pd.read_parquet(A/"acron_stundenwerte_wert.parquet")
imp      = pd.read_parquet(A/"Import_Data_stadtbach_15min.parquet").set_index("Datum")

print("ACRON:", acron_raw.shape, acron_raw.index.min(), "→", acron_raw.index.max())
print("Index eindeutig:", acron_raw.index.is_unique, "| Frequenz stündlich:",
      (acron_raw.index.to_series().diff().dropna() == pd.Timedelta("1h")).mean().round(4))
missing_h = pd.date_range(f"{CFG['year']}-01-01", f"{CFG['year']}-12-31 23:00", freq="h").difference(acron_raw.index)
print("Fehlende Stunden im Jahr 2025:", list(missing_h), "(30.03. 02:00 existiert lokal nicht – Sommerzeitbeginn; die doppelte Stunde 26.10. 02:00 wurde beim Upload auf den ersten Wert reduziert; Zeitstempel naiv lokal)")
print("Import 15 min:", imp.shape, imp.index.min(), "→", imp.index.max(), "| eindeutig:", imp.index.is_unique)

def kanal(name):
    n = name
    for k,v in [("Druck_VL","p_VL"),("Druck_RL","p_RL"),("DiffDruck","dp"),("Temp_VL","T_VL"),("Temp_RL","T_RL"),
                ("Durchfluss","m_dot"),("Waermeleistung","Q"),("Drehzahl","n_pumpe")]:
        if k in n: return v
    return "sonstige"
meta = pd.DataFrame({"kanal": [kanal(c) for c in acron_raw.columns]}, index=acron_raw.columns)
meta = meta.join(sig.set_index("key")[["ordner","title"]])
print(meta.kanal.value_counts().to_dict())

In [ ]:
# --- 6.2 Qualitätsanalyse + Bereinigung (regelbasiert, dokumentiert)
def longest_const_run(s):
    s = s.dropna()
    if s.empty: return 0
    return int((s.diff() != 0).cumsum().value_counts().max())

acron = acron_raw.copy()
log = []
p_cols = [c for c in acron.columns if meta.kanal[c] in ("p_VL","p_RL","dp")]
for c in p_cols:
    s = acron[c]
    nb = pd.concat([s.shift(1), s.shift(-1)], axis=1).median(axis=1)         # Nachbarn ±1 h
    if meta.kanal[c] == "dp":
        bad = (s <= 0.0) & (nb > 0.5)
    else:
        bad = (s < 0.5) & (nb > 2.0)                                          # Nullwerte bei intakten Nachbarn
    if bad.any():
        acron.loc[bad, c] = np.nan; log.append((c, "Druck/Δp ≈ 0 bei intakten Nachbarn → NaN", int(bad.sum())))
t_cols = [c for c in acron.columns if meta.kanal[c] in ("T_VL","T_RL")]
for c in t_cols:
    s = acron[c]; bad = (s < 5) | (s > 135)
    if bad.any(): acron.loc[bad, c] = np.nan; log.append((c, "Temperatur außerhalb 5…135 °C → NaN", int(bad.sum())))
m_cols = [c for c in acron.columns if meta.kanal[c] == "m_dot"]
for c in m_cols:
    if c.startswith("HKW_Durchfluss_SL"): continue     # Vorzeichen = Flussrichtung (bewusst erlaubt)
    s = acron[c]; bad = s < -1e-9
    if bad.any(): acron.loc[bad, c] = np.nan; log.append((c, "negativer Durchfluss → NaN", int(bad.sum())))

cleanlog = pd.DataFrame(log, columns=["signal","regel","anzahl"])
print(f"Bereinigungsregeln angewandt auf {len(cleanlog)} Signale, {cleanlog.anzahl.sum()} Werte → NaN")

rows=[]
for c in acron.columns:
    s0, s = acron_raw[c], acron[c]
    rows.append(dict(signal=c, kanal=meta.kanal[c], station=meta.ordner[c],
        abdeckung_pct=round(100*s.notna().mean(),1), luecken_pct=round(100*s.isna().mean(),1),
        min=s.min(), q05=s.quantile(.05), median=s.median(), q95=s.quantile(.95), max=s.max(),
        nullwerte_roh=int((s0==0).sum()), laengster_konst_lauf_h=longest_const_run(s0),
        ausreisser_3sigma=int(((s-s.mean()).abs() > 3*s.std()).sum()) if s.std()>0 else 0,
        bereinigt=int(s0.notna().sum()-s.notna().sum())))
qual = pd.DataFrame(rows).set_index("signal")
qual["eingefroren_verdacht"] = (qual.laengster_konst_lauf_h >= 72) & ~qual.kanal.isin(["n_pumpe"])
# Validierungstauglichkeit: Abdeckung ≥ 80 % und kein Stillstand ≥ 168 h (Pumpendrehzahlen ausgenommen, dort ist Stillstand normal)
qual["validierungstauglich"] = (qual.abdeckung_pct >= 80) & ((qual.laengster_konst_lauf_h < 168) | qual.kanal.isin(["n_pumpe"]))
qual.round(2).to_csv(RESULTS/"datenqualitaet_acron.csv")
cleanlog.to_csv(RESULTS/"bereinigungslog.csv", index=False)

### 6.3 Ergebnis der Qualitätsprüfung

In [ ]:
print("Abdeckung je Kanal (Median über Signale, %):")
print(qual.groupby("kanal").abdeckung_pct.agg(["count","median","min"]).round(1))
print("\nSignale mit < 80 % Abdeckung:")
print(qual[qual.abdeckung_pct < 80][["kanal","station","abdeckung_pct"]].sort_values("abdeckung_pct").to_string())
print("\nEingefrorene Reihen (≥ 72 h konstant, Verdacht auf Sensor-/Übertragungsausfall):")
print(qual[qual.eingefroren_verdacht][["kanal","station","laengster_konst_lauf_h"]].sort_values("laengster_konst_lauf_h", ascending=False).to_string())
print("\nValidierungstauglich (Abdeckung ≥ 80 %, kein Stillstand ≥ 168 h):", int(qual.validierungstauglich.sum()), "von", len(qual), "Signalen; je Kanal:", qual.groupby("kanal").validierungstauglich.sum().to_dict())
print("\nBereinigung (Top 15):"); print(cleanlog.sort_values("anzahl", ascending=False).head(15).to_string(index=False))

### 6.4 Einheiten- und Konsistenzprüfung
* **Durchfluss:** Aus `Q = V̇·cp·ρ·ΔT` wird an Stationen mit Leistung, Durchfluss, VL- und RL-Temperatur die implizite Einheit geprüft (Erwartung: m³/h).
* **Druck:** VL > RL, Δp ≈ VL − RL (soweit alle drei Signale vorhanden).

In [ ]:
# Einheitenprüfung Durchfluss: Q[MW] ≈ V[m³/h]·ρ·cp·ΔT/3.6e6 (ρ≈960, cp≈4,2 kJ/kgK)
rows=[]
for st in sorted(set(meta.ordner)):
    q  = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="Q"]
    v  = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="m_dot"]
    tv = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="T_VL"]
    tr = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="T_RL"]
    if q and v and tv and tr:
        d = pd.concat([acron[q[0]], acron[v[0]], acron[tv[0]]-acron[tr[0]]], axis=1, keys=["Q","V","dT"]).dropna()
        d = d[(d.Q>0.05)&(d.V>5)&(d.dT>5)]
        if len(d) > 100:
            q_calc = d.V*960*4.2*d.dT/3.6e6
            rows.append((st, len(d), round((d.Q/q_calc).median(),3), round((d.Q/q_calc).quantile(.1),2), round((d.Q/q_calc).quantile(.9),2)))
einh = pd.DataFrame(rows, columns=["station","n","Q_gemessen/Q_aus_V·ΔT (Median)","q10","q90"])
print(einh.to_string(index=False))
print("\n→ Kundenstationen: Verhältnis ≈ 1,04 (Näherungskonstanten ρ=960, cp=4,2) bestätigt m³/h. Abweichend: HKW (≈15×), HWW sekundär (≈2,2×), GT-Ost (≈0,7×) – dort ist Definition von Durchfluss bzw. Leistung (Teilstrom/Bezugsstelle) ungeklärt → Rückfrage; diese Signale werden nicht als Validierungsziel verwendet.")

# Druckkonsistenz
pc = []
for st in sorted(set(meta.ordner)):
    pv = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="p_VL"]
    pr = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="p_RL"]
    dd = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="dp"]
    if pv and pr:
        d = (acron[pv[0]]-acron[pr[0]]).dropna()
        row = dict(station=st, VL_kleiner_RL_pct=round(100*(d<0).mean(),1), dp_VL_RL_median=round(d.median(),2))
        if dd:
            e = (acron[dd[0]]-(acron[pv[0]]-acron[pr[0]])).dropna(); row["dp_gemessen_minus_(VL−RL)_median"]=round(e.median(),2)
        pc.append(row)
print(); print(pd.DataFrame(pc).to_string(index=False))

## 7. Explorative Datenanalyse
Außentemperatur, Netztemperaturen, Drücke am HKW und Differenzdrücke an den Kundenstationen 2025.

In [ ]:
Ta = imp["outdoor_temp_C"].resample("1h").mean()
Ta = Ta.reindex(acron.index)
fig, ax = plt.subplots(4,1, figsize=(12,10), sharex=True)
ax[0].plot(Ta, color=COL["grau"], lw=.6); ax[0].set_ylabel("Außentemp. [°C]")
ax[1].plot(acron["HKW_Temp_VL"], color=COL["vl"], lw=.6, label="HKW Vorlauf"); ax[1].plot(acron["HKW_Temp_RL"], color=COL["rl"], lw=.6, label="HKW Rücklauf")
ax[1].set_ylabel("Temperatur [°C]"); ax[1].legend(loc="upper right", ncol=2)
ax[2].plot(acron["HKW_Druck_VL"], color=COL["vl"], lw=.6, label="HKW Druck VL"); ax[2].plot(acron["HKW_Druck_RL"], color=COL["rl"], lw=.6, label="HKW Druck RL")
ax[2].set_ylabel("Druck [bar Ü]"); ax[2].legend(loc="upper right", ncol=2)
ax[3].plot(acron["HKW_Durchfluss_gesamt"], color=COL["mess"], lw=.6); ax[3].set_ylabel("HKW Durchfluss ges. [m³/h]")
fig.suptitle("Netzzustand am HKW, 2025", y=.995); fig.tight_layout(); fig.savefig(FIG_DIR/"07_hkw_zeitreihen.png"); plt.show()

In [ ]:
# Differenzdruck-Dauerlinien der Kundenstationen: kritische Stellen
dp_cols = [c for c in acron.columns if meta.kanal[c]=="dp" and meta.ordner[c] not in ("PSS","PSW","HKW","BMHKW","GT-Ost","AVA","Heizwerk_Sued","HWW_prim_Einsp")]
dpq = pd.DataFrame({c:{"q01":acron[c].quantile(.01),"median":acron[c].median(),"anteil_<0,7bar_%":100*(acron[c]<0.7).mean()} for c in dp_cols}).T.round(2)
dpq = dpq.sort_values("q01")
display(dpq.head(12))
fig, ax = plt.subplots(figsize=(9,5))
for c in dpq.index[:6]:
    s = acron[c].dropna().sort_values(ascending=False).reset_index(drop=True)
    ax.plot(np.linspace(0,100,len(s)), s, label=c.replace("_DiffDruck",""))
ax.axhline(0.7, color=COL["warn"], ls="--", label="0,7 bar (YAML-Annahme)")
ax.set_xlabel("Zeitanteil [%]"); ax.set_ylabel("Differenzdruck [bar]"); ax.set_title("Differenzdruck-Dauerlinien, sechs kritischste Stationen"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR/"07_dp_dauerlinien.png"); plt.show()

## 8. Prüfung der Heizkurve
Vergleich von (a) der **Vorgabe** (Auftrag), (b) der **YAML-Kurve**, (c) der **gemessenen** Vorlauftemperatur am Erzeugeraustritt und (d) der **statistisch aus Messdaten** ermittelten Kurve.

Vorgabe: $T_{VL,soll}=\min\big(120\,^\circ\mathrm{C},\ \max(90\,^\circ\mathrm{C},\ 110\,^\circ\mathrm{C}-T_{a})\big)$
YAML: linear von 122 °C bei −15 °C bis 70 °C bei +30 °C.

In [ ]:
def hk_vorgabe(Ta):
    return np.clip(110.0 - np.asarray(Ta, float), 90.0, 120.0)
def hk_yaml(Ta):
    return np.clip(122.0 - (122.0-70.0)/(30.0-(-15.0))*(np.asarray(Ta, float)+15.0), 70.0, 122.0)

# Tests: Endpunkte
assert hk_vorgabe(-10)==120 and hk_vorgabe(20)==90 and hk_vorgabe(-20)==120 and hk_vorgabe(30)==90
assert abs(hk_yaml(-15)-122)<1e-9 and abs(hk_yaml(30)-70)<1e-9
print("Heizkurven-Endpunkttests bestanden.")

hk = pd.DataFrame({"Ta":Ta, "VL_HKW":acron["HKW_Temp_VL"], "VL_MAN":acron.get("MAN_Temp_VL_Station"),
                   "VL_PPS":acron.get("PPS_Temp_VL"), "VL_PSW":acron.get("PSW_Temp_VL"),
                   "V_ges":acron["HKW_Durchfluss_gesamt"]}).dropna(subset=["Ta","VL_HKW"])
hk["soll_vorgabe"] = hk_vorgabe(hk.Ta); hk["soll_yaml"] = hk_yaml(hk.Ta)
hk["mon"] = hk.index.month; hk["stunde"] = hk.index.hour
hk["saison"] = hk.mon.map(lambda m: "Winter" if m in (12,1,2) else "Frühjahr" if m in (3,4,5) else "Sommer" if m in (6,7,8) else "Herbst")
hk["last"] = pd.qcut(hk.V_ges, 3, labels=["niedrig","mittel","hoch"])

# Statistische Kurve: Median/Quantile je 2-K-Klasse + lineare Anpassung (−10…+20 °C)
hk["bin"] = (np.floor(hk.Ta/2)*2+1)
st = hk.groupby("bin").VL_HKW.agg(n="count", q10=lambda s: s.quantile(.1), median="median", q90=lambda s: s.quantile(.9)).query("n>=20")
sel = st.loc[(st.index>=-10)&(st.index<=20)]
k, d0 = np.polyfit(sel.index, sel["median"], 1)
print(f"Statistische Kurve (Median, −10…20 °C): T_VL = {d0:.1f} {k:+.3f}·T_a  → bei −10 °C: {d0+k*-10:.1f} °C, bei +20 °C: {d0+k*20:.1f} °C")
for nm, f in (("Vorgabe",hk_vorgabe),("YAML",hk_yaml)):
    r = hk.VL_HKW - f(hk.Ta); print(f"{nm:8s}: Bias {r.mean():+.1f} K, RMSE {np.sqrt((r**2).mean()):.1f} K, MAE {r.abs().mean():.1f} K")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,5))
xs = np.linspace(-13,32,100)
ax[0].scatter(hk.Ta, hk.VL_HKW, s=3, alpha=.15, color=COL["grau"], label="Messung HKW, stündlich")
ax[0].fill_between(st.index, st.q10, st.q90, color=COL["vl"], alpha=.15, label="10–90 %-Band")
ax[0].plot(st.index, st["median"], "o-", color=COL["vl"], ms=3, label="Median je 2-K-Klasse")
ax[0].plot(xs, hk_vorgabe(xs), color=COL["ok"], lw=2, label="Vorgabe (Auftrag)")
ax[0].plot(xs, hk_yaml(xs), color=COL["warn"], lw=2, ls="--", label="YAML-Kurve")
ax[0].set(xlabel="Außentemperatur [°C]", ylabel="Vorlauftemperatur [°C]", ylim=(60,130), title="Heizkurve: Vorgabe, YAML und Messung")
ax[0].legend(fontsize=8)
res = (hk.VL_HKW - hk.soll_vorgabe)
hk.assign(res=res).boxplot(column="res", by="saison", ax=ax[1], grid=False)
ax[1].axhline(0, color="k", lw=.8); ax[1].set(title="Abweichung Messung − Vorgabe nach Jahreszeit", xlabel="", ylabel="ΔT_VL [K]"); plt.suptitle("")
fig.tight_layout(); fig.savefig(FIG_DIR/"08_heizkurve.png"); plt.show()

In [ ]:
# Abweichungen nach Tageszeit, Lastniveau, Netzbereich
hk["res"] = hk.VL_HKW - hk.soll_vorgabe
print("Nach Tageszeit (Median Abweichung, K):")
print(hk.groupby(pd.cut(hk.stunde,[-1,5,11,17,23],labels=["Nacht 0–5","Vormittag 6–11","Nachmittag 12–17","Abend 18–23"]),observed=True).res.median().round(1).to_string())
print("\nNach Lastniveau (HKW-Durchfluss-Terzile):"); print(hk.groupby("last",observed=True).res.median().round(1).to_string())
bereich = pd.DataFrame({b: (hk[c] - hk.soll_vorgabe) for b,c in (("HKW (Mitte)","VL_HKW"),("MAN","VL_MAN"),("Pumpstation Süd","VL_PPS"),("Pumpstation West","VL_PSW")) if hk[c].notna().any()})
print("\nNach Netzbereich (Median Abweichung zur Vorgabe, K; Winter):")
print(bereich[hk.saison=="Winter"].median().round(1).to_string())
kalt = hk[hk.Ta < -5]
print(f"\nKältester Bereich (T_a < −5 °C, n={len(kalt)} h): gemessener Vorlauf Median {kalt.VL_HKW.median():.1f} °C, max {kalt.VL_HKW.max():.1f} °C; Vorgabe 115–120 °C")
print("Spitzen ≥ 120 °C:", int((hk.VL_HKW>=120).sum()), "Stunden; ≥ 122 °C:", int((hk.VL_HKW>=122).sum()), "Stunden")

### 8.4 Zwischenbewertung Heizkurve (aus den Rechenergebnissen oben)

In [ ]:
rm = {n: float(np.sqrt(((hk.VL_HKW - f(hk.Ta))**2).mean())) for n, f in (("Vorgabe", hk_vorgabe), ("YAML", hk_yaml))}
bias = {n: float((hk.VL_HKW - f(hk.Ta)).mean()) for n, f in (("Vorgabe", hk_vorgabe), ("YAML", hk_yaml))}
best = min(rm, key=rm.get)
print(f"Hauptrechnung: Heizkurve '{best}' (RMSE {rm[best]:.1f} K, Bias {bias[best]:+.1f} K) gegenüber '{[k for k in rm if k!=best][0]}' (RMSE {max(rm.values()):.1f} K, Bias {bias[[k for k in rm if k!=best][0]]:+.1f} K).")
print(f"Statistische Kurve T_VL = {d0:.1f} {k:+.3f}·T_a weicht bei −10 °C um {d0+k*-10-120:+.1f} K und bei +20 °C um {d0+k*20-90:+.1f} K von der Vorgabe ab.")
print("Die YAML-Kurve wird als Sensitivität mitgeführt (Konflikt K-01). Es gab 2025 keinen Speicherbetrieb; eine Abhängigkeit vom Speicher entfällt.")
print("Hinweis: Nur 182 Stunden mit T_a < −5 °C und minimal −12,9 °C im Jahr 2025 – der Auslegungsfall (−14 … −16 °C) ist NICHT durch Messung abgedeckt und wird durch die Kurve extrapoliert.")

---
**Ende Etappe 1.** Folgt: Kapitel 9–10 (Netzpläne, kanonisches Netzmodell) – danach Halt zur Prüfung der abgelesenen Topologie.